# Statistical Significance Analysis for Classical Models

This notebook performs a dataset-level statistical analysis of the classical-model predictions.

- Main analysis: `test` split
- Complementary analysis: `val` split
- Statistical unit: `dataset_id`
- Metric: macro-F1 per `dataset_id × model_id`
- Global test: Friedman test
- Post-hoc test: Wilcoxon signed-rank test with Holm-Bonferroni correction

All outputs are saved automatically to `codes/results/classical_models/statistical_tests/`.

In [8]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Optional
import warnings

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.stats import friedmanchisquare, wilcoxon

warnings.simplefilter("always", UserWarning)

ALPHA = 0.05
FEW_DATASETS_WARNING_THRESHOLD = 8
REQUIRED_COLUMNS = [
    "split",
    "embedding_model",
    "classifier",
    "model",
    "idx",
    "dataset_id",
    "true_label_id",
    "pred_label_id",
]


def find_project_root(start: Optional[Path] = None) -> Path:
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "codes" / "results" / "classical_models").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the project root containing codes/results/classical_models."
    )


PROJECT_ROOT = find_project_root()
CLASSICAL_RESULTS_DIR = PROJECT_ROOT / "codes" / "results" / "classical_models"
OUTPUT_DIR = CLASSICAL_RESULTS_DIR / "statistical_tests"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEST_PATH = CLASSICAL_RESULTS_DIR / "classical_models_test_predictions.csv"
VAL_PATH = CLASSICAL_RESULTS_DIR / "classical_models_val_predictions.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")

Project root: C:\Users\usuario\OneDrive\Documentos\UFG_Remoto\Projects\_ANATEL\TRE\papers-repos\wics-csbd2026
Output directory: C:\Users\usuario\OneDrive\Documentos\UFG_Remoto\Projects\_ANATEL\TRE\papers-repos\wics-csbd2026\codes\results\classical_models\statistical_tests


In [9]:
def load_predictions(path: Path, expected_split: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    missing_columns = sorted(set(REQUIRED_COLUMNS) - set(df.columns))
    if missing_columns:
        raise ValueError(
            f"{path.name} is missing required columns: {missing_columns}"
        )

    df = df.copy()
    df["split"] = df["split"].astype(str).str.strip().str.lower()

    split_mask = df["split"].eq(expected_split.lower())
    n_other_splits = int((~split_mask).sum())
    if n_other_splits:
        warnings.warn(
            f"{path.name}: {n_other_splits} rows do not belong to split "
            f"'{expected_split}' and will be ignored."
        )
    df = df.loc[split_mask].copy()

    key_columns = [
        "dataset_id",
        "embedding_model",
        "classifier",
        "true_label_id",
        "pred_label_id",
    ]
    missing_rows = int(df[key_columns].isna().any(axis=1).sum())
    if missing_rows:
        warnings.warn(
            f"{path.name}: {missing_rows} rows have missing values in essential "
            "columns and will be dropped."
        )
        df = df.dropna(subset=key_columns).copy()

    for column in ["dataset_id", "embedding_model", "classifier", "model", "idx"]:
        df[column] = df[column].astype(str).str.strip()

    df["model_id"] = df["embedding_model"] + " + " + df["classifier"]
    return df


def macro_f1_score(y_true: pd.Series, y_pred: pd.Series) -> float:
    y_true = pd.Series(y_true).reset_index(drop=True)
    y_pred = pd.Series(y_pred).reset_index(drop=True)
    labels = pd.Index(pd.unique(pd.concat([y_true, y_pred], ignore_index=True)))

    if len(labels) == 0:
        return np.nan

    per_label_f1 = []
    for label in labels:
        true_positive = int(((y_true == label) & (y_pred == label)).sum())
        false_positive = int(((y_true != label) & (y_pred == label)).sum())
        false_negative = int(((y_true == label) & (y_pred != label)).sum())

        precision_den = true_positive + false_positive
        recall_den = true_positive + false_negative
        precision = true_positive / precision_den if precision_den else 0.0
        recall = true_positive / recall_den if recall_den else 0.0

        if precision + recall == 0.0:
            per_label_f1.append(0.0)
        else:
            per_label_f1.append(2 * precision * recall / (precision + recall))

    return float(np.mean(per_label_f1))


def compute_macro_f1_by_dataset_model(df: pd.DataFrame) -> pd.DataFrame:
    records = []
    grouped = df.groupby(["dataset_id", "model_id"], dropna=False, sort=True)

    for (dataset_id, model_id), group in grouped:
        macro_f1 = macro_f1_score(group["true_label_id"], group["pred_label_id"])
        records.append(
            {
                "dataset_id": dataset_id,
                "model_id": model_id,
                "macro_f1": float(macro_f1),
                "n_samples": int(len(group)),
            }
        )

    macro_f1_df = pd.DataFrame(records)
    if macro_f1_df.empty:
        raise ValueError("No dataset-model pairs were available to compute macro-F1.")

    return macro_f1_df


def pivot_macro_f1(macro_f1_df: pd.DataFrame) -> pd.DataFrame:
    pivot = (
        macro_f1_df.pivot(index="dataset_id", columns="model_id", values="macro_f1")
        .sort_index()
        .sort_index(axis=1)
    )
    return pivot


def holm_bonferroni(p_values: np.ndarray, alpha: float = ALPHA) -> tuple[np.ndarray, np.ndarray]:
    p_values = np.asarray(p_values, dtype=float)
    n_tests = len(p_values)

    if n_tests == 0:
        return np.array([], dtype=float), np.array([], dtype=bool)

    order = np.argsort(p_values)
    adjusted = np.full(n_tests, np.nan, dtype=float)
    running_max = 0.0

    for rank, idx in enumerate(order):
        multiplier = n_tests - rank
        adjusted_value = min(1.0, p_values[idx] * multiplier)
        running_max = max(running_max, adjusted_value)
        adjusted[idx] = running_max

    reject = np.zeros(n_tests, dtype=bool)
    for rank, idx in enumerate(order):
        threshold = alpha / (n_tests - rank)
        if p_values[idx] <= threshold:
            reject[idx] = True
        else:
            break

    return adjusted, reject


def compute_model_average_ranking(
    pivot: pd.DataFrame, complete_cases: pd.DataFrame
) -> pd.DataFrame:
    all_ranks = pivot.rank(axis=1, ascending=False, method="average")
    complete_ranks = (
        complete_cases.rank(axis=1, ascending=False, method="average")
        if not complete_cases.empty
        else pd.DataFrame(index=[], columns=pivot.columns, dtype=float)
    )

    ranking_df = pd.DataFrame(
        {
            "model_id": pivot.columns,
            "mean_macro_f1": pivot.mean(axis=0, skipna=True).reindex(pivot.columns).values,
            "std_macro_f1": pivot.std(axis=0, ddof=0, skipna=True).reindex(pivot.columns).values,
            "median_macro_f1": pivot.median(axis=0, skipna=True).reindex(pivot.columns).values,
            "avg_rank_all_available": all_ranks.mean(axis=0, skipna=True).reindex(pivot.columns).values,
            "avg_rank_complete_cases": complete_ranks.mean(axis=0, skipna=True).reindex(pivot.columns).values,
            "n_datasets_available": pivot.notna().sum(axis=0).reindex(pivot.columns).values,
        }
    )

    ranking_df = ranking_df.sort_values(
        ["mean_macro_f1", "avg_rank_all_available", "model_id"],
        ascending=[False, True, True],
        kind="mergesort",
    ).reset_index(drop=True)
    ranking_df.insert(0, "mean_macro_f1_rank", np.arange(1, len(ranking_df) + 1))
    return ranking_df

In [10]:
def run_friedman_test(complete_cases: pd.DataFrame, split_name: str) -> pd.DataFrame:
    n_datasets, n_models = complete_cases.shape
    result = {
        "split": split_name,
        "n_models": int(n_models),
        "n_complete_datasets": int(n_datasets),
        "alpha": ALPHA,
        "statistic": np.nan,
        "p_value": np.nan,
        "kendall_w": np.nan,
        "significant": False,
        "status": "not_run",
        "note": "",
    }

    if n_models < 3:
        result["status"] = "skipped"
        result["note"] = "Friedman test requires at least three models."
        return pd.DataFrame([result])

    if n_datasets < 2:
        result["status"] = "skipped"
        result["note"] = "Friedman test requires at least two complete datasets."
        return pd.DataFrame([result])

    statistic, p_value = friedmanchisquare(
        *[complete_cases[column].to_numpy() for column in complete_cases.columns]
    )
    kendall_w = statistic / (n_datasets * (n_models - 1))

    result.update(
        {
            "statistic": float(statistic),
            "p_value": float(p_value),
            "kendall_w": float(kendall_w),
            "significant": bool(p_value < ALPHA),
            "status": "ok",
        }
    )
    return pd.DataFrame([result])


def run_pairwise_wilcoxon(
    pivot: pd.DataFrame, best_model: str, split_name: str
) -> pd.DataFrame:
    records = []
    low_pair_count = 0

    for comparison_model in pivot.columns:
        if comparison_model == best_model:
            continue

        paired = pivot[[best_model, comparison_model]].dropna()
        n_paired = int(len(paired))
        diff = paired[best_model] - paired[comparison_model]

        record = {
            "split": split_name,
            "reference_model": best_model,
            "comparison_model": comparison_model,
            "n_paired_datasets": n_paired,
            "mean_macro_f1_reference": float(paired[best_model].mean()) if n_paired else np.nan,
            "mean_macro_f1_comparison": float(paired[comparison_model].mean()) if n_paired else np.nan,
            "mean_diff_reference_minus_comparison": float(diff.mean()) if n_paired else np.nan,
            "median_diff_reference_minus_comparison": float(diff.median()) if n_paired else np.nan,
            "statistic": np.nan,
            "raw_p_value": np.nan,
            "holm_adjusted_p_value": np.nan,
            "reject_h0_holm": False,
            "alpha": ALPHA,
            "status": "not_run",
            "note": "",
        }

        if 0 < n_paired < FEW_DATASETS_WARNING_THRESHOLD:
            low_pair_count += 1

        if n_paired < 2:
            record["status"] = "skipped"
            record["note"] = "Wilcoxon test requires at least two paired datasets."
            records.append(record)
            continue

        non_zero_diff = ~np.isclose(diff.to_numpy(), 0.0, atol=1e-12)
        if not np.any(non_zero_diff):
            record["statistic"] = 0.0
            record["raw_p_value"] = 1.0
            record["status"] = "all_zero_differences"
            record["note"] = "All paired macro-F1 differences were zero."
            records.append(record)
            continue

        try:
            statistic, p_value = wilcoxon(
                paired[best_model].to_numpy(),
                paired[comparison_model].to_numpy(),
                alternative="two-sided",
                zero_method="wilcox",
                method="auto",
            )
            record["statistic"] = float(statistic)
            record["raw_p_value"] = float(p_value)
            record["status"] = "ok"
        except ValueError as exc:
            record["status"] = "error"
            record["note"] = str(exc)

        records.append(record)

    posthoc_df = pd.DataFrame(records)
    valid_mask = posthoc_df["raw_p_value"].notna()
    adjusted, reject = holm_bonferroni(
        posthoc_df.loc[valid_mask, "raw_p_value"].to_numpy(),
        alpha=ALPHA,
    )
    posthoc_df.loc[valid_mask, "holm_adjusted_p_value"] = adjusted
    posthoc_df.loc[valid_mask, "reject_h0_holm"] = reject

    if low_pair_count:
        warnings.warn(
            f"{split_name}: {low_pair_count} pairwise Wilcoxon comparisons used fewer "
            f"than {FEW_DATASETS_WARNING_THRESHOLD} paired datasets."
        )

    posthoc_df = posthoc_df.sort_values(
        ["raw_p_value", "comparison_model"],
        ascending=[True, True],
        na_position="last",
        kind="mergesort",
    ).reset_index(drop=True)
    return posthoc_df


def format_p_value(value: float) -> str:
    if pd.isna(value):
        return "NA"
    return f"{value:.4g}"


def build_methodology_text(alpha: float = ALPHA) -> str:
    return (
        "To assess whether the observed differences among the classical models were "
        "statistically significant, we treated each dataset as the statistical unit and "
        "computed one macro-F1 score for every dataset-model pair. The main analysis "
        "was performed on the test split, while the validation split was analyzed only "
        "as a complementary check. We first applied the Friedman test to compare all "
        "models jointly across datasets. We then selected the model with the highest "
        "mean macro-F1 across datasets and compared it against each remaining model "
        "using two-sided Wilcoxon signed-rank tests. The resulting p-values were "
        "adjusted with the Holm-Bonferroni correction to control the family-wise error "
        f"rate. All statistical decisions used a significance level of alpha = {alpha:.2f}. "
        "This design avoids treating individual predictions as independent observations "
        "and instead bases the inference on dataset-level performance."
    )


def build_result_text(
    split_name: str,
    best_model: str,
    ranking_df: pd.DataFrame,
    friedman_df: pd.DataFrame,
    posthoc_df: pd.DataFrame,
) -> str:
    best_row = ranking_df.loc[ranking_df["model_id"].eq(best_model)].iloc[0]
    friedman_row = friedman_df.iloc[0]
    significant_df = posthoc_df.loc[posthoc_df["reject_h0_holm"].fillna(False)].copy()

    if friedman_row["status"] != "ok":
        return (
            f"On the {split_name} split, the model with the highest mean macro-F1 was "
            f"`{best_model}` (mean macro-F1 = {best_row['mean_macro_f1']:.4f}). "
            "However, the global Friedman test could not be completed because there "
            f"were only {int(friedman_row['n_complete_datasets'])} complete datasets "
            f"for {int(friedman_row['n_models'])} models."
        )

    if len(significant_df):
        sig_phrase = (
            f"{len(significant_df)} of {len(posthoc_df)} post-hoc comparisons remained "
            "significant after Holm-Bonferroni correction"
        )
    else:
        sig_phrase = (
            f"none of the {len(posthoc_df)} post-hoc comparisons remained significant "
            "after Holm-Bonferroni correction"
        )

    return (
        f"On the {split_name} split, the Friedman test compared "
        f"{int(friedman_row['n_models'])} models across "
        f"{int(friedman_row['n_complete_datasets'])} complete datasets using "
        f"dataset-level macro-F1 values (chi-square = {friedman_row['statistic']:.4f}, "
        f"p = {format_p_value(friedman_row['p_value'])}). The model with the highest "
        f"mean macro-F1 was `{best_model}` (mean macro-F1 = {best_row['mean_macro_f1']:.4f}); "
        f"{sig_phrase} at alpha = {ALPHA:.2f}."
    )


def run_split_analysis(df: pd.DataFrame, split_name: str) -> Dict[str, object]:
    macro_f1_df = compute_macro_f1_by_dataset_model(df)
    pivot = pivot_macro_f1(macro_f1_df)
    complete_cases = pivot.dropna(axis=0, how="any")

    notes: List[str] = []
    incomplete_datasets = int(pivot.isna().any(axis=1).sum())
    if incomplete_datasets:
        note = (
            f"{split_name}: {incomplete_datasets} datasets do not contain all models and "
            "were excluded from the Friedman test."
        )
        notes.append(note)
        warnings.warn(note)

    if len(complete_cases) < FEW_DATASETS_WARNING_THRESHOLD:
        note = (
            f"{split_name}: only {len(complete_cases)} complete datasets are available "
            "for the Friedman test, so statistical power may be limited."
        )
        notes.append(note)
        warnings.warn(note)

    ranking_df = compute_model_average_ranking(pivot, complete_cases)
    best_model = ranking_df.iloc[0]["model_id"]

    friedman_df = run_friedman_test(complete_cases, split_name)
    friedman_df.insert(2, "n_total_datasets", int(pivot.shape[0]))
    friedman_df.insert(3, "n_excluded_datasets_due_to_missing_models", incomplete_datasets)

    posthoc_df = run_pairwise_wilcoxon(pivot, best_model, split_name)
    methodology_text = build_methodology_text(alpha=ALPHA)
    result_text = build_result_text(
        split_name=split_name,
        best_model=best_model,
        ranking_df=ranking_df,
        friedman_df=friedman_df,
        posthoc_df=posthoc_df,
    )

    pivot.to_csv(OUTPUT_DIR / f"{split_name}_macro_f1_by_dataset_model.csv")
    ranking_df.to_csv(OUTPUT_DIR / f"{split_name}_model_average_ranking.csv", index=False)
    friedman_df.to_csv(OUTPUT_DIR / f"{split_name}_friedman_result.csv", index=False)
    posthoc_df.to_csv(OUTPUT_DIR / f"{split_name}_wilcoxon_posthoc_holm.csv", index=False)

    return {
        "macro_f1_df": macro_f1_df,
        "pivot": pivot,
        "complete_cases": complete_cases,
        "ranking_df": ranking_df,
        "friedman_df": friedman_df,
        "posthoc_df": posthoc_df,
        "best_model": best_model,
        "methodology_text": methodology_text,
        "result_text": result_text,
        "notes": notes,
    }


def build_summary_markdown(analyses: Dict[str, Dict[str, object]]) -> str:
    lines = [
        "# Statistical Significance Summary",
        "",
        "This file was generated automatically by `codes/4-estatistic-result.ipynb`.",
        "",
    ]

    for split_name in ["test", "val"]:
        analysis = analyses.get(split_name)
        if analysis is None:
            continue

        friedman_row = analysis["friedman_df"].iloc[0]
        ranking_df = analysis["ranking_df"]
        best_model = analysis["best_model"]
        best_row = ranking_df.loc[ranking_df["model_id"].eq(best_model)].iloc[0]
        posthoc_df = analysis["posthoc_df"]
        significant_models = posthoc_df.loc[
            posthoc_df["reject_h0_holm"].fillna(False), "comparison_model"
        ].tolist()

        lines.extend(
            [
                f"## {split_name.title()} split",
                "",
                f"- Datasets available: {analysis['pivot'].shape[0]}",
                f"- Models available: {analysis['pivot'].shape[1]}",
                f"- Best mean macro-F1 model: `{best_model}` ({best_row['mean_macro_f1']:.4f})",
                f"- Friedman status: {friedman_row['status']}",
                f"- Friedman p-value: {format_p_value(friedman_row['p_value'])}",
                f"- Complete datasets used in Friedman: {int(friedman_row['n_complete_datasets'])}",
            ]
        )

        if significant_models:
            lines.append(
                "- Models significantly different from the best model after Holm-Bonferroni: "
                + ", ".join(f"`{model}`" for model in significant_models)
            )
        else:
            lines.append(
                "- Models significantly different from the best model after Holm-Bonferroni: none"
            )

        if analysis["notes"]:
            lines.append("- Notes:")
            for note in analysis["notes"]:
                lines.append(f"  - {note}")

        lines.extend(
            [
                "",
                "### Paper-ready results sentence",
                "",
                analysis["result_text"],
                "",
            ]
        )

    lines.extend(
        [
            "## Paper-ready methodology text",
            "",
            analyses["test"]["methodology_text"],
            "",
        ]
    )
    return "\n".join(lines)

In [11]:
test_df = load_predictions(TEST_PATH, expected_split="test")
val_df = load_predictions(VAL_PATH, expected_split="val")

overview_df = pd.DataFrame(
    [
        {
            "split": "test",
            "rows": len(test_df),
            "datasets": test_df["dataset_id"].nunique(),
            "model_ids": test_df["model_id"].nunique(),
        },
        {
            "split": "val",
            "rows": len(val_df),
            "datasets": val_df["dataset_id"].nunique(),
            "model_ids": val_df["model_id"].nunique(),
        },
    ]
)

display(Markdown("## Loaded data overview"))
display(overview_df)

## Loaded data overview

,split,rows,datasets,model_ids
0,test,200992,10,32
1,val,200992,10,32


In [12]:
test_analysis = run_split_analysis(test_df, split_name="test")

display(Markdown("## Test split results"))
display(Markdown(f"**Best mean macro-F1 model:** `{test_analysis['best_model']}`"))
display(Markdown("### Macro-F1 by dataset and model"))
display(test_analysis["pivot"].round(4))
display(Markdown("### Model average ranking"))
display(test_analysis["ranking_df"].round(4))
display(Markdown("### Friedman test"))
display(test_analysis["friedman_df"].round(6))
display(Markdown("### Wilcoxon post-hoc tests with Holm-Bonferroni correction"))
display(test_analysis["posthoc_df"].round(6))

## Test split results

**Best mean macro-F1 model:** `bertimbau_large + svc_rbf`

### Macro-F1 by dataset and model

model_id,albertina_ptbr_100m + decision_tree,albertina_ptbr_100m + gaussian_nb,albertina_ptbr_100m + gradient_boosting,albertina_ptbr_100m + knn,albertina_ptbr_100m + linearsvc,albertina_ptbr_100m + logreg,albertina_ptbr_100m + random_forest,albertina_ptbr_100m + svc_rbf,albertina_ptbr_900m + decision_tree,albertina_ptbr_900m + gaussian_nb,...,bertimbau_base + random_forest,bertimbau_base + svc_rbf,bertimbau_large + decision_tree,bertimbau_large + gaussian_nb,bertimbau_large + gradient_boosting,bertimbau_large + knn,bertimbau_large + linearsvc,bertimbau_large + logreg,bertimbau_large + random_forest,bertimbau_large + svc_rbf
dataset_id,,,,,,,,,,,,,,,,,,,,,
CENTRALFATOS,0.4836,0.4987,0.4976,0.4954,0.5315,0.5315,0.4981,0.4972,0.4877,0.4989,...,0.4976,0.4979,0.4869,0.4989,0.4972,0.5564,0.4963,0.4969,0.4982,0.5817
COVID19BR,0.5572,0.5181,0.6220,0.6608,0.6120,0.6176,0.6184,0.7705,0.5408,0.5350,...,0.6128,0.7973,0.5515,0.5912,0.5864,0.6805,0.6396,0.6484,0.6373,0.7700
FACTCKBR,0.6704,0.4459,0.6907,0.9319,0.5942,0.5748,0.8854,0.8915,0.7188,0.4402,...,0.8854,0.8854,0.7152,0.4205,0.5919,0.9127,0.6355,0.6228,0.8854,0.8854
FAKEBR,0.6005,0.3337,0.7422,0.7877,0.7913,0.7941,0.6655,0.9240,0.6843,0.3349,...,0.7443,0.9315,0.6599,0.3632,0.8558,0.8399,0.8823,0.8776,0.7740,0.9491
FAKETRUEBR,0.7072,0.2333,0.8800,0.8357,0.8881,0.8919,0.8537,0.9497,0.7328,0.2308,...,0.8677,0.9403,0.7241,0.2907,0.8690,0.8137,0.8658,0.8674,0.8714,0.9515
FCN,0.8093,0.3326,0.8852,0.9643,0.9546,0.9546,0.8748,0.9806,0.8211,0.3397,...,0.9118,0.9806,0.7875,0.3483,0.9382,0.9546,0.9449,0.9449,0.9051,0.9773
FNEWSSET,0.4038,0.4498,0.4404,0.3603,0.4498,0.4564,0.4034,0.6606,0.4341,0.4886,...,0.4251,0.7129,0.3529,0.4949,0.4371,0.3844,0.6457,0.6129,0.4034,0.6890
FRECOGNA,0.7335,0.4226,0.8382,0.8670,0.8303,0.8303,0.8707,0.9042,0.7676,0.4226,...,0.8673,0.9255,0.7272,0.4226,0.8392,0.8402,0.8465,0.8341,0.8628,0.9199
MUMINPT,0.5267,0.4152,0.5361,0.5410,0.5273,0.5305,0.6311,0.5608,0.4922,0.3328,...,0.6016,0.5272,0.4171,0.2953,0.6186,0.5901,0.5175,0.5428,0.6311,0.5852


### Model average ranking

,mean_macro_f1_rank,model_id,mean_macro_f1,std_macro_f1,median_macro_f1,avg_rank_all_available,avg_rank_complete_cases,n_datasets_available
0,1,bertimbau_large + svc_rbf,0.8243,0.1474,0.9026,3.50,3.50,10
1,2,albertina_ptbr_900m + svc_rbf,0.8219,0.1463,0.8897,2.50,2.50,10
2,3,bertimbau_base + svc_rbf,0.8065,0.1643,0.8757,6.00,6.00,10
3,4,albertina_ptbr_100m + svc_rbf,0.7914,0.1608,0.8332,6.80,6.80,10
4,5,albertina_ptbr_900m + linearsvc,0.7479,0.1603,0.7726,10.20,10.20,10
5,6,bertimbau_large + knn,0.7439,0.1741,0.8268,11.85,11.85,10
6,7,albertina_ptbr_900m + knn,0.7389,0.1922,0.8297,12.20,12.20,10
7,8,albertina_ptbr_900m + logreg,0.7381,0.1708,0.7818,10.15,10.15,10
8,9,bertimbau_base + knn,0.7346,0.1633,0.8048,13.15,13.15,10
9,10,bertimbau_large + linearsvc,0.7339,0.1560,0.7461,14.55,14.55,10


### Friedman test

,split,n_models,n_total_datasets,n_excluded_datasets_due_to_missing_models,n_complete_datasets,alpha,statistic,p_value,kendall_w,significant,status,note
0,test,32,10,0,10,0.05,161.953064,0.0,0.522429,True,ok,


### Wilcoxon post-hoc tests with Holm-Bonferroni correction

,split,reference_model,comparison_model,n_paired_datasets,mean_macro_f1_reference,mean_macro_f1_comparison,mean_diff_reference_minus_comparison,median_diff_reference_minus_comparison,statistic,raw_p_value,holm_adjusted_p_value,reject_h0_holm,alpha,status,note
0,test,bertimbau_large + svc_rbf,albertina_ptbr_100m + decision_tree,10,0.824253,0.617338,0.206915,0.213934,0.0,0.001953,0.060547,False,0.05,ok,
1,test,bertimbau_large + svc_rbf,albertina_ptbr_100m + gaussian_nb,10,0.824253,0.392247,0.432006,0.468412,0.0,0.001953,0.060547,False,0.05,ok,
2,test,bertimbau_large + svc_rbf,albertina_ptbr_100m + gradient_boosting,10,0.824253,0.695231,0.129022,0.102749,0.0,0.001953,0.060547,False,0.05,ok,
3,test,bertimbau_large + svc_rbf,albertina_ptbr_100m + linearsvc,10,0.824253,0.684309,0.139943,0.123674,0.0,0.001953,0.060547,False,0.05,ok,
4,test,bertimbau_large + svc_rbf,albertina_ptbr_100m + logreg,10,0.824253,0.689042,0.135211,0.120998,0.0,0.001953,0.060547,False,0.05,ok,
5,test,bertimbau_large + svc_rbf,albertina_ptbr_900m + decision_tree,10,0.824253,0.631501,0.192752,0.192702,0.0,0.001953,0.060547,False,0.05,ok,
6,test,bertimbau_large + svc_rbf,albertina_ptbr_900m + gaussian_nb,10,0.824253,0.386733,0.437520,0.471228,0.0,0.001953,0.060547,False,0.05,ok,
7,test,bertimbau_large + svc_rbf,albertina_ptbr_900m + gradient_boosting,10,0.824253,0.706705,0.117548,0.091232,0.0,0.001953,0.060547,False,0.05,ok,
8,test,bertimbau_large + svc_rbf,albertina_ptbr_900m + linearsvc,10,0.824253,0.747864,0.076389,0.075003,0.0,0.001953,0.060547,False,0.05,ok,
9,test,bertimbau_large + svc_rbf,albertina_ptbr_900m + logreg,10,0.824253,0.738085,0.086168,0.080256,0.0,0.001953,0.060547,False,0.05,ok,


In [13]:
val_analysis = run_split_analysis(val_df, split_name="val")

display(Markdown("## Validation split results (complementary)"))
display(Markdown(f"**Best mean macro-F1 model:** `{val_analysis['best_model']}`"))
display(Markdown("### Macro-F1 by dataset and model"))
display(val_analysis["pivot"].round(4))
display(Markdown("### Model average ranking"))
display(val_analysis["ranking_df"].round(4))
display(Markdown("### Friedman test"))
display(val_analysis["friedman_df"].round(6))
display(Markdown("### Wilcoxon post-hoc tests with Holm-Bonferroni correction"))
display(val_analysis["posthoc_df"].round(6))

## Validation split results (complementary)

**Best mean macro-F1 model:** `bertimbau_large + svc_rbf`

### Macro-F1 by dataset and model

model_id,albertina_ptbr_100m + decision_tree,albertina_ptbr_100m + gaussian_nb,albertina_ptbr_100m + gradient_boosting,albertina_ptbr_100m + knn,albertina_ptbr_100m + linearsvc,albertina_ptbr_100m + logreg,albertina_ptbr_100m + random_forest,albertina_ptbr_100m + svc_rbf,albertina_ptbr_900m + decision_tree,albertina_ptbr_900m + gaussian_nb,...,bertimbau_base + random_forest,bertimbau_base + svc_rbf,bertimbau_large + decision_tree,bertimbau_large + gaussian_nb,bertimbau_large + gradient_boosting,bertimbau_large + knn,bertimbau_large + linearsvc,bertimbau_large + logreg,bertimbau_large + random_forest,bertimbau_large + svc_rbf
dataset_id,,,,,,,,,,,,,,,,,,,,,
CENTRALFATOS,0.4836,0.4984,0.4963,0.4969,0.5188,0.4939,0.5817,0.5401,0.4886,0.4984,...,0.4981,0.4979,0.4989,0.4984,0.4977,0.4964,0.5288,0.5330,0.4982,0.5564
COVID19BR,0.5301,0.5072,0.6555,0.6460,0.6598,0.6569,0.6341,0.7977,0.5825,0.5570,...,0.5965,0.7521,0.5741,0.5852,0.6332,0.7083,0.6607,0.6567,0.6098,0.7897
FACTCKBR,0.6828,0.4442,0.6897,0.9082,0.5887,0.5887,0.8854,0.8915,0.6786,0.4254,...,0.8854,0.9341,0.5907,0.4240,0.5730,0.8976,0.6278,0.5970,0.8748,0.9154
FAKEBR,0.6330,0.3415,0.7515,0.7783,0.8254,0.8254,0.6815,0.9055,0.6816,0.3427,...,0.7563,0.9454,0.6569,0.3826,0.8516,0.8550,0.8859,0.8869,0.7860,0.9454
FAKETRUEBR,0.7087,0.2360,0.8425,0.8173,0.8525,0.8469,0.8385,0.9422,0.7405,0.2373,...,0.8446,0.9271,0.6919,0.2729,0.8217,0.8305,0.8303,0.8191,0.8578,0.9403
FCN,0.7278,0.3305,0.8987,0.9515,0.9288,0.9385,0.8301,0.9806,0.8100,0.3333,...,0.8819,0.9677,0.7677,0.3362,0.8920,0.9580,0.9225,0.9289,0.8785,0.9774
FNEWSSET,0.4390,0.4994,0.3847,0.3541,0.6172,0.6315,0.3576,0.5250,0.3603,0.5640,...,0.3576,0.6092,0.4800,0.6154,0.4421,0.3908,0.6229,0.5792,0.3750,0.6092
FRECOGNA,0.7387,0.4203,0.8588,0.8731,0.8593,0.8588,0.8761,0.9249,0.7701,0.4203,...,0.8880,0.9373,0.7471,0.4192,0.8818,0.8490,0.8661,0.8633,0.8958,0.9462
MUMINPT,0.4612,0.4466,0.5468,0.6164,0.4496,0.4475,0.4816,0.5608,0.4581,0.3058,...,0.6016,0.6001,0.4422,0.2752,0.5266,0.6024,0.5235,0.5036,0.5403,0.5648


### Model average ranking

,mean_macro_f1_rank,model_id,mean_macro_f1,std_macro_f1,median_macro_f1,avg_rank_all_available,avg_rank_complete_cases,n_datasets_available
0,1,bertimbau_large + svc_rbf,0.8110,0.1616,0.8902,2.85,2.85,10
1,2,bertimbau_base + svc_rbf,0.7969,0.1649,0.8628,5.30,5.30,10
2,3,albertina_ptbr_100m + svc_rbf,0.7777,0.1708,0.8446,5.95,5.95,10
3,4,albertina_ptbr_900m + svc_rbf,0.7760,0.1798,0.8286,7.50,7.50,10
4,5,bertimbau_large + knn,0.7386,0.1761,0.8145,12.35,12.35,10
5,6,bertimbau_base + knn,0.7342,0.1809,0.8361,13.15,13.15,10
6,7,bertimbau_large + linearsvc,0.7288,0.1442,0.7403,11.60,11.60,10
7,8,albertina_ptbr_900m + linearsvc,0.7288,0.1543,0.7056,11.30,11.30,10
8,9,albertina_ptbr_100m + knn,0.7217,0.1812,0.7766,14.95,14.95,10
9,10,bertimbau_large + logreg,0.7211,0.1542,0.7379,12.75,12.75,10


### Friedman test

,split,n_models,n_total_datasets,n_excluded_datasets_due_to_missing_models,n_complete_datasets,alpha,statistic,p_value,kendall_w,significant,status,note
0,val,32,10,0,10,0.05,141.433484,0.0,0.456237,True,ok,


### Wilcoxon post-hoc tests with Holm-Bonferroni correction

,split,reference_model,comparison_model,n_paired_datasets,mean_macro_f1_reference,mean_macro_f1_comparison,mean_diff_reference_minus_comparison,median_diff_reference_minus_comparison,statistic,raw_p_value,holm_adjusted_p_value,reject_h0_holm,alpha,status,note
0,val,bertimbau_large + svc_rbf,albertina_ptbr_100m + decision_tree,10,0.810979,0.610657,0.200322,0.219555,0.0,0.001953,0.060547,False,0.05,ok,
1,val,bertimbau_large + svc_rbf,albertina_ptbr_100m + gaussian_nb,10,0.810979,0.392045,0.418934,0.498565,0.0,0.001953,0.060547,False,0.05,ok,
2,val,bertimbau_large + svc_rbf,albertina_ptbr_100m + gradient_boosting,10,0.810979,0.678030,0.132949,0.116025,0.0,0.001953,0.060547,False,0.05,ok,
3,val,bertimbau_large + svc_rbf,albertina_ptbr_900m + decision_tree,10,0.810979,0.632538,0.178441,0.187975,0.0,0.001953,0.060547,False,0.05,ok,
4,val,bertimbau_large + svc_rbf,albertina_ptbr_900m + gaussian_nb,10,0.810979,0.386603,0.424376,0.507954,0.0,0.001953,0.060547,False,0.05,ok,
5,val,bertimbau_large + svc_rbf,albertina_ptbr_900m + gradient_boosting,10,0.810979,0.695584,0.115395,0.092777,0.0,0.001953,0.060547,False,0.05,ok,
6,val,bertimbau_large + svc_rbf,albertina_ptbr_900m + random_forest,10,0.810979,0.695438,0.115541,0.058540,0.0,0.001953,0.060547,False,0.05,ok,
7,val,bertimbau_large + svc_rbf,bertimbau_base + decision_tree,10,0.810979,0.594961,0.216018,0.217434,0.0,0.001953,0.060547,False,0.05,ok,
8,val,bertimbau_large + svc_rbf,bertimbau_base + gaussian_nb,10,0.810979,0.407923,0.403056,0.509905,0.0,0.001953,0.060547,False,0.05,ok,
9,val,bertimbau_large + svc_rbf,bertimbau_base + gradient_boosting,10,0.810979,0.671512,0.139467,0.108560,0.0,0.001953,0.060547,False,0.05,ok,


In [14]:
analyses = {"test": test_analysis, "val": val_analysis}
summary_markdown = build_summary_markdown(analyses)
summary_path = OUTPUT_DIR / "statistical_significance_summary.md"
summary_path.write_text(summary_markdown, encoding="utf-8")

display(Markdown("## Paper-ready methodology text"))
display(Markdown(test_analysis["methodology_text"]))

display(Markdown("## Paper-ready results sentence (test split)"))
display(Markdown(test_analysis["result_text"]))

display(Markdown("## Paper-ready results sentence (validation split)"))
display(Markdown(val_analysis["result_text"]))

display(Markdown("## Saved files"))
for path in sorted(OUTPUT_DIR.glob("*")):
    print(path.relative_to(PROJECT_ROOT))

print(f"\nSummary saved to: {summary_path}")

## Paper-ready methodology text

To assess whether the observed differences among the classical models were statistically significant, we treated each dataset as the statistical unit and computed one macro-F1 score for every dataset-model pair. The main analysis was performed on the test split, while the validation split was analyzed only as a complementary check. We first applied the Friedman test to compare all models jointly across datasets. We then selected the model with the highest mean macro-F1 across datasets and compared it against each remaining model using two-sided Wilcoxon signed-rank tests. The resulting p-values were adjusted with the Holm-Bonferroni correction to control the family-wise error rate. All statistical decisions used a significance level of alpha = 0.05. This design avoids treating individual predictions as independent observations and instead bases the inference on dataset-level performance.

## Paper-ready results sentence (test split)

On the test split, the Friedman test compared 32 models across 10 complete datasets using dataset-level macro-F1 values (chi-square = 161.9531, p = 1.156e-19). The model with the highest mean macro-F1 was `bertimbau_large + svc_rbf` (mean macro-F1 = 0.8243); none of the 31 post-hoc comparisons remained significant after Holm-Bonferroni correction at alpha = 0.05.

## Paper-ready results sentence (validation split)

On the val split, the Friedman test compared 32 models across 10 complete datasets using dataset-level macro-F1 values (chi-square = 141.4335, p = 4.776e-16). The model with the highest mean macro-F1 was `bertimbau_large + svc_rbf` (mean macro-F1 = 0.8110); none of the 31 post-hoc comparisons remained significant after Holm-Bonferroni correction at alpha = 0.05.

## Saved files

codes\results\classical_models\statistical_tests\statistical_significance_summary.md
codes\results\classical_models\statistical_tests\test_classifier_breakdown_summary.csv
codes\results\classical_models\statistical_tests\test_classifier_embedding_macro_f1_detail.csv
codes\results\classical_models\statistical_tests\test_classifier_embedding_mean_macro_f1.csv
codes\results\classical_models\statistical_tests\test_classifier_embedding_mean_macro_f1_heatmap.png
codes\results\classical_models\statistical_tests\test_classifier_embedding_rank_within_classifier.csv
codes\results\classical_models\statistical_tests\test_classifier_embedding_std_macro_f1.csv
codes\results\classical_models\statistical_tests\test_embedding_vs_classifier_spread.csv
codes\results\classical_models\statistical_tests\test_embedding_vs_classifier_spread_summary.csv
codes\results\classical_models\statistical_tests\test_friedman_result.csv
codes\results\classical_models\statistical_tests\test_macro_f1_by_dataset_model.csv
c

In [15]:
# Supplementary classifier-level breakdown requested by the reviewer.
def build_classifier_breakdown(df: pd.DataFrame, split_name: str) -> Dict[str, object]:
    records = []
    grouped = df.groupby(["dataset_id", "classifier", "embedding_model"], sort=True)

    for (dataset_id, classifier, embedding_model), group in grouped:
        records.append(
            {
                "dataset_id": dataset_id,
                "classifier": classifier,
                "embedding_model": embedding_model,
                "model_id": f"{embedding_model} + {classifier}",
                "macro_f1": macro_f1_score(group["true_label_id"], group["pred_label_id"]),
                "n_samples": int(len(group)),
            }
        )

    detail_df = (
        pd.DataFrame(records)
        .sort_values(["classifier", "embedding_model", "dataset_id"], kind="mergesort")
        .reset_index(drop=True)
    )

    mean_table = (
        detail_df.pivot_table(
            index="classifier",
            columns="embedding_model",
            values="macro_f1",
            aggfunc="mean",
        )
        .sort_index()
        .sort_index(axis=1)
    )
    std_table = (
        detail_df.pivot_table(
            index="classifier",
            columns="embedding_model",
            values="macro_f1",
            aggfunc="std",
        )
        .sort_index()
        .sort_index(axis=1)
    )
    rank_table = mean_table.rank(axis=1, ascending=False, method="average")

    def tied_labels(row: pd.Series, reducer: str = "max") -> str:
        reference = row.max() if reducer == "max" else row.min()
        mask = np.isclose(row.to_numpy(dtype=float), reference, atol=1e-12)
        return " | ".join(row.index[mask].tolist())

    classifier_summary = (
        detail_df.groupby("classifier")["macro_f1"]
        .agg(
            mean_macro_f1="mean",
            std_macro_f1=lambda s: float(np.std(s, ddof=0)),
            median_macro_f1="median",
            n_dataset_embedding_pairs="count",
        )
        .sort_index()
    )
    classifier_summary["best_embedding_by_mean"] = mean_table.apply(
        lambda row: tied_labels(row, reducer="max"), axis=1
    )
    classifier_summary["worst_embedding_by_mean"] = mean_table.apply(
        lambda row: tied_labels(row, reducer="min"), axis=1
    )
    classifier_summary["mean_embedding_gap"] = mean_table.max(axis=1) - mean_table.min(axis=1)
    classifier_summary["mean_rank_across_embeddings"] = rank_table.mean(axis=1)
    classifier_summary = (
        classifier_summary.sort_values(
            ["mean_macro_f1", "mean_embedding_gap", "classifier"],
            ascending=[False, False, True],
            kind="mergesort",
        )
        .reset_index()
    )

    embedding_spread_by_dataset = (
        detail_df.groupby(["dataset_id", "classifier"])["macro_f1"]
        .agg(lambda s: float(s.max() - s.min()))
        .groupby("dataset_id")
        .mean()
    )
    classifier_spread_by_dataset = (
        detail_df.groupby(["dataset_id", "embedding_model"])["macro_f1"]
        .agg(lambda s: float(s.max() - s.min()))
        .groupby("dataset_id")
        .mean()
    )
    spread_df = (
        pd.DataFrame(
            {
                "mean_embedding_spread_within_classifier": embedding_spread_by_dataset,
                "mean_classifier_spread_within_embedding": classifier_spread_by_dataset,
            }
        )
        .reset_index()
        .sort_values("dataset_id", kind="mergesort")
        .reset_index(drop=True)
    )
    spread_df["embedding_minus_classifier_spread"] = (
        spread_df["mean_embedding_spread_within_classifier"]
        - spread_df["mean_classifier_spread_within_embedding"]
    )

    spread_summary = pd.DataFrame(
        [
            {
                "split": split_name,
                "avg_embedding_spread_within_classifier": float(
                    spread_df["mean_embedding_spread_within_classifier"].mean()
                ),
                "avg_classifier_spread_within_embedding": float(
                    spread_df["mean_classifier_spread_within_embedding"].mean()
                ),
                "embedding_minus_classifier_spread": float(
                    spread_df["embedding_minus_classifier_spread"].mean()
                ),
                "supports_embedding_matters_more": bool(
                    spread_df["embedding_minus_classifier_spread"].mean() > 0
                ),
            }
        ]
    )

    detail_df.to_csv(OUTPUT_DIR / f"{split_name}_classifier_embedding_macro_f1_detail.csv", index=False)
    mean_table.to_csv(OUTPUT_DIR / f"{split_name}_classifier_embedding_mean_macro_f1.csv")
    std_table.to_csv(OUTPUT_DIR / f"{split_name}_classifier_embedding_std_macro_f1.csv")
    rank_table.to_csv(OUTPUT_DIR / f"{split_name}_classifier_embedding_rank_within_classifier.csv")
    classifier_summary.to_csv(OUTPUT_DIR / f"{split_name}_classifier_breakdown_summary.csv", index=False)
    spread_df.to_csv(OUTPUT_DIR / f"{split_name}_embedding_vs_classifier_spread.csv", index=False)
    spread_summary.to_csv(OUTPUT_DIR / f"{split_name}_embedding_vs_classifier_spread_summary.csv", index=False)

    figure_path = OUTPUT_DIR / f"{split_name}_classifier_embedding_mean_macro_f1_heatmap.png"
    figure_status = "not attempted"
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt

        fig_width = max(8, 1.8 * len(mean_table.columns))
        fig_height = max(4, 0.9 * len(mean_table.index))
        fig, ax = plt.subplots(figsize=(fig_width, fig_height))
        image = ax.imshow(mean_table.to_numpy(), cmap="YlGnBu", aspect="auto")
        ax.set_xticks(range(len(mean_table.columns)))
        ax.set_xticklabels(mean_table.columns, rotation=45, ha="right")
        ax.set_yticks(range(len(mean_table.index)))
        ax.set_yticklabels(mean_table.index)
        ax.set_title(f"{split_name.title()} split: mean macro-F1 by classifier and embedding")

        for row_idx in range(mean_table.shape[0]):
            for col_idx in range(mean_table.shape[1]):
                value = mean_table.iat[row_idx, col_idx]
                ax.text(col_idx, row_idx, f"{value:.3f}", ha="center", va="center", fontsize=8)

        colorbar = fig.colorbar(image, ax=ax)
        colorbar.set_label("Mean macro-F1")
        fig.tight_layout()
        fig.savefig(figure_path, dpi=200, bbox_inches="tight")
        plt.close(fig)
        figure_status = f"saved to {figure_path.relative_to(PROJECT_ROOT)}"
    except Exception as exc:
        warnings.warn(
            f"{split_name}: could not generate the classifier heatmap automatically: {exc}"
        )
        figure_status = f"not saved ({exc})"

    summary_text = (
        f"On the {split_name} split, the average spread across embeddings within the same "
        f"classifier was {spread_summary.iloc[0]['avg_embedding_spread_within_classifier']:.4f}, "
        f"whereas the average spread across classifiers within the same embedding was "
        f"{spread_summary.iloc[0]['avg_classifier_spread_within_embedding']:.4f}."
    )

    return {
        "detail_df": detail_df,
        "mean_table": mean_table,
        "std_table": std_table,
        "rank_table": rank_table,
        "classifier_summary": classifier_summary,
        "spread_df": spread_df,
        "spread_summary": spread_summary,
        "summary_text": summary_text,
        "figure_status": figure_status,
    }


test_classifier_breakdown = build_classifier_breakdown(test_df, split_name="test")
val_classifier_breakdown = build_classifier_breakdown(val_df, split_name="val")

display(Markdown("## Supplementary classifier-level comparison"))
display(
    Markdown(
        "This addendum reports all classifiers individually, rather than focusing only on "
        "the best-performing configuration, so the reviewer can inspect the embedding effect "
        "inside each classifier."
    )
)

display(Markdown("### Test split: mean macro-F1 by classifier and embedding"))
display(test_classifier_breakdown["mean_table"].round(4))
display(Markdown("### Test split: embedding ranks within each classifier"))
display(test_classifier_breakdown["rank_table"].round(4))
display(Markdown("### Test split: classifier summary"))
display(test_classifier_breakdown["classifier_summary"].round(4))
display(Markdown("### Test split: embedding vs classifier spread"))
display(test_classifier_breakdown["spread_df"].round(4))
display(Markdown(test_classifier_breakdown["summary_text"]))

display(Markdown("### Validation split: mean macro-F1 by classifier and embedding"))
display(val_classifier_breakdown["mean_table"].round(4))
display(Markdown("### Validation split: embedding ranks within each classifier"))
display(val_classifier_breakdown["rank_table"].round(4))
display(Markdown("### Validation split: classifier summary"))
display(val_classifier_breakdown["classifier_summary"].round(4))
display(Markdown("### Validation split: embedding vs classifier spread"))
display(val_classifier_breakdown["spread_df"].round(4))
display(Markdown(val_classifier_breakdown["summary_text"]))

display(Markdown("### Supplementary files generated"))
for split_name in ["test", "val"]:
    print(f"{split_name}:")
    for suffix in [
        "classifier_embedding_macro_f1_detail.csv",
        "classifier_embedding_mean_macro_f1.csv",
        "classifier_embedding_std_macro_f1.csv",
        "classifier_embedding_rank_within_classifier.csv",
        "classifier_breakdown_summary.csv",
        "embedding_vs_classifier_spread.csv",
        "embedding_vs_classifier_spread_summary.csv",
        "classifier_embedding_mean_macro_f1_heatmap.png",
    ]:
        print(f"  - {(OUTPUT_DIR / f'{split_name}_{suffix}').relative_to(PROJECT_ROOT)}")
    print(
        f"  - heatmap status: "
        f"{test_classifier_breakdown['figure_status'] if split_name == 'test' else val_classifier_breakdown['figure_status']}"
    )

## Supplementary classifier-level comparison

This addendum reports all classifiers individually, rather than focusing only on the best-performing configuration, so the reviewer can inspect the embedding effect inside each classifier.

### Test split: mean macro-F1 by classifier and embedding

embedding_model,albertina_ptbr_100m,albertina_ptbr_900m,bertimbau_base,bertimbau_large
classifier,,,,
decision_tree,0.6173,0.6315,0.6066,0.5902
gaussian_nb,0.3922,0.3867,0.4025,0.3939
gradient_boosting,0.6952,0.7067,0.6684,0.6915
knn,0.7155,0.7389,0.7346,0.7439
linearsvc,0.6843,0.7479,0.6954,0.7339
logreg,0.6890,0.7381,0.6941,0.7313
random_forest,0.7070,0.7114,0.7110,0.7158
svc_rbf,0.7914,0.8219,0.8065,0.8243


### Test split: embedding ranks within each classifier

...

### Validation split: embedding vs classifier spread

,dataset_id,mean_embedding_spread_within_classifier,mean_classifier_spread_within_embedding,embedding_minus_classifier_spread
0,CENTRALFATOS,0.0295,0.0488,-0.0192
1,COVID19BR,0.0655,0.2524,-0.1868
2,FACTCKBR,0.0629,0.4850,-0.4222
3,FAKEBR,0.0689,0.5832,-0.5143
4,FAKETRUEBR,0.0441,0.6774,-0.6333
5,FCN,0.0486,0.6415,-0.5929
6,FNEWSSET,0.0938,0.2528,-0.1590
7,FRECOGNA,0.0248,0.5156,-0.4908
8,MUMINPT,0.0968,0.2389,-0.1421
9,TRE300,0.1682,0.6133,-0.4450


On the val split, the average spread across embeddings within the same classifier was 0.0703, whereas the average spread across classifiers within the same embedding was 0.4309.

### Supplementary files generated

test:
  - codes\results\classical_models\statistical_tests\test_classifier_embedding_macro_f1_detail.csv
  - codes\results\classical_models\statistical_tests\test_classifier_embedding_mean_macro_f1.csv
  - codes\results\classical_models\statistical_tests\test_classifier_embedding_std_macro_f1.csv
  - codes\results\classical_models\statistical_tests\test_classifier_embedding_rank_within_classifier.csv
  - codes\results\classical_models\statistical_tests\test_classifier_breakdown_summary.csv
  - codes\results\classical_models\statistical_tests\test_embedding_vs_classifier_spread.csv
  - codes\results\classical_models\statistical_tests\test_embedding_vs_classifier_spread_summary.csv
  - codes\results\classical_models\statistical_tests\test_classifier_embedding_mean_macro_f1_heatmap.png
  - heatmap status: saved to codes\results\classical_models\statistical_tests\test_classifier_embedding_mean_macro_f1_heatmap.png
val:
  - codes\results\classical_models\statistical_tests\val_classifier_emb